# ML-08 — Capstone Modeling Lane

Method: Logistic Regression, Random Forest, and Gradient Boosting with 5-fold grouped CV.
Goal: predict 30-day impression growth and rank pages for review.


## 1. Method choice and why

This is a binary classification task with an observed future label. The final decision is a ranked list, so every model outputs a probability and we sort by it.

- **Logistic Regression**: readable linear model; coefficients show direction and magnitude.
- **Random Forest**: captures non-linear interactions and provides feature importances.
- **Gradient Boosting** (GradientBoostingClassifier): often the strongest tabular learner; gives a third comparison point.

Primary metric: **Precision@50**, inspects a fixed top-K list.  
Secondary metrics: ROC-AUC, Average Precision, Precision@10, Precision@100.  
Validation: **5-fold grouped cross-validation** by `client_hash_id` so no client leaks between train and test.


In [1]:
# Method summary
print('Method: Logistic Regression + Random Forest + Gradient Boosting')
print('Task: binary classification -> ranking by growth probability')
print('Primary metric: Precision@50')
print('Validation: 5-fold StratifiedGroupKFold by client_hash_id')


Method: Logistic Regression + Random Forest + Gradient Boosting
Task: binary classification -> ranking by growth probability
Primary metric: Precision@50
Validation: 5-fold StratifiedGroupKFold by client_hash_id


## 2. Split design

All rows share the same feature window (2026-01-01 to 2026-03-31) and the same label window (2026-04-01 to 2026-04-30), so we cannot split by time. Instead we use grouped cross-validation.

- 5 folds, grouped by `client_hash_id`.
- Stratified by `growth_label` so each fold keeps roughly the same positive rate.
- Every prediction is out-of-fold: the model never saw that client during training.
- This simulates deploying on unseen clients and averages out the noise from any single test group.

We also cache the feature vector to disk so repeated runs do not re-scan the warehouse.


In [2]:
%pip install -q duckdb pandas scikit-learn matplotlib

import os
import json
import getpass
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import duckdb

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight

# Load .env if present
env_path = Path.cwd().parents[1] / '.env'
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ.setdefault(key, value)

cache_path = Path.cwd().parent / 'outputs' / 'model_feature_vector.csv'

if cache_path.exists():
    feature_vector = pd.read_csv(cache_path)
    print(f'Loaded cached feature vector: {len(feature_vector):,} rows')
else:
    token = os.getenv('HF_TOKEN')
    if not token:
        token = getpass.getpass('Hugging Face token: ')

    conn = duckdb.connect()
    conn.execute('INSTALL httpfs;')
    conn.execute('LOAD httpfs;')
    conn.execute(f'''
        CREATE OR REPLACE SECRET hf_token (
            TYPE HTTP,
            EXTRA_HTTP_HEADERS MAP {{
                'Authorization': 'Bearer {token}'
            }}
        );
    ''')

    conn.execute('''
        CREATE OR REPLACE VIEW dim_content AS
        SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
    ''')

    fv_query = '''
        WITH feature_window AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS gsc_impressions_90d,
                AVG(gsc_avg_position) AS gsc_avg_position_90d,
                SUM(gsc_clicks) AS gsc_clicks_90d,
                CASE WHEN SUM(gsc_impressions) > 0 THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions) ELSE 0 END AS gsc_ctr_90d,
                SUM(CASE WHEN ga4_data_available IS TRUE THEN sessions_organic ELSE 0 END) AS sessions_organic_90d,
                COUNT(DISTINCT report_date) AS days_with_impressions_90d
            FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-0[1-3]/*.parquet')
            WHERE report_date BETWEEN '2026-01-01' AND '2026-03-31'
            GROUP BY client_hash_id, content_hash_id
        ),
        last_30 AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS gsc_impressions_last_30d
            FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
            WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
            GROUP BY client_hash_id, content_hash_id
        ),
        label_window AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS gsc_impressions_next_30d
            FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet')
            WHERE report_date BETWEEN '2026-04-01' AND '2026-04-30'
            GROUP BY client_hash_id, content_hash_id
        ),
        decision_date AS (
            SELECT DATE '2026-03-31' AS d
        )
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            f.gsc_impressions_90d,
            f.gsc_avg_position_90d,
            f.gsc_clicks_90d,
            f.gsc_ctr_90d,
            f.sessions_organic_90d,
            f.days_with_impressions_90d,
            l30.gsc_impressions_last_30d,
            l.gsc_impressions_next_30d,
            dim_content.search_volume,
            dim_content.competition,
            dim_content.cpc,
            dim_content.word_count,
            dim_content.content_type,
            dim_content.main_intent,
            (d.d - dim_content.content_created_date)::INTEGER AS content_age_days,
            CASE
                WHEN l.gsc_impressions_next_30d >= 1.20 * l30.gsc_impressions_last_30d
                     AND l30.gsc_impressions_last_30d >= 100
                THEN 1 ELSE 0
            END AS growth_label
        FROM feature_window f
        LEFT JOIN last_30 l30 ON f.content_hash_id = l30.content_hash_id
        LEFT JOIN label_window l ON f.content_hash_id = l.content_hash_id
        LEFT JOIN dim_content ON f.content_hash_id = dim_content.content_hash_id
        CROSS JOIN decision_date d
        WHERE f.gsc_impressions_90d >= 300
          AND dim_content.content_created_date <= d.d
          AND dim_content.word_count IS NOT NULL
    '''

    feature_vector = conn.execute(fv_query).df()
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    feature_vector.to_csv(cache_path, index=False)
    print(f'Built and cached feature vector: {len(feature_vector):,} rows')

print(f'Overall base growth rate: {feature_vector["growth_label"].mean():.3%}')


Note: you may need to restart the kernel to use updated packages.


Loaded cached feature vector: 67,478 rows
Overall base growth rate: 24.731%


In [3]:
# Prepare modeling frame
df = feature_vector.copy()

# Fill missing intent and build the same recent_share used by the baseline
df['main_intent'] = df['main_intent'].fillna('unknown')
df['recent_share'] = (df['gsc_impressions_last_30d'] / df['gsc_impressions_90d']).clip(0, 1).fillna(0)

# Engineered features: log transforms and the interaction the baseline rule already uses
# All inputs are strictly pre-decision
df['log_gsc_impressions_90d'] = np.log1p(df['gsc_impressions_90d'])
df['log_gsc_impressions_last_30d'] = np.log1p(df['gsc_impressions_last_30d'])
df['momentum_x_volume'] = df['recent_share'] * df['log_gsc_impressions_90d']

# Keep a raw copy for error analysis (categoricals intact)
df_raw = df.copy()

numeric_features = [
    'gsc_impressions_90d', 'gsc_avg_position_90d', 'gsc_clicks_90d', 'gsc_ctr_90d',
    'sessions_organic_90d', 'days_with_impressions_90d', 'gsc_impressions_last_30d',
    'search_volume', 'competition', 'cpc', 'word_count', 'content_age_days',
    'recent_share', 'log_gsc_impressions_90d', 'log_gsc_impressions_last_30d', 'momentum_x_volume'
]
categorical_features = ['content_type', 'main_intent']

# One-hot encode categoricals
df = pd.get_dummies(df, columns=categorical_features, prefix=categorical_features, dtype=int)

feature_cols = numeric_features + [
    c for c in df.columns
    if c.startswith('content_type_') or c.startswith('main_intent_')
]

# Sanity check: no leakage columns in feature list
leakage_cols = {'growth_label', 'gsc_impressions_next_30d', 'client_hash_id', 'content_hash_id'}
assert len(set(feature_cols) & leakage_cols) == 0, f'Leakage columns in features: {set(feature_cols) & leakage_cols}'

X = df[feature_cols].fillna(0).replace([np.inf, -np.inf], 0)
y = df['growth_label'].values
groups = df['client_hash_id'].values

print(f'Features ({len(feature_cols)}):')
for c in feature_cols:
    print(f'  - {c}')


Features (24):
  - gsc_impressions_90d
  - gsc_avg_position_90d
  - gsc_clicks_90d
  - gsc_ctr_90d
  - sessions_organic_90d
  - days_with_impressions_90d
  - gsc_impressions_last_30d
  - search_volume
  - competition
  - cpc
  - word_count
  - content_age_days
  - recent_share
  - log_gsc_impressions_90d
  - log_gsc_impressions_last_30d
  - momentum_x_volume
  - content_type_comparison article
  - content_type_feedly article
  - content_type_keyword article
  - main_intent_commercial
  - main_intent_informational
  - main_intent_navigational
  - main_intent_transactional
  - main_intent_unknown


## 3. Train + compare vs my baseline

### 3.1. 5-fold grouped cross-validation

For every fold we:
1. Recompute the baseline rule on the held-out test clients.
2. Train Logistic Regression, Random Forest, and Gradient Boosting on the training clients.
3. Score every model on the held-out test clients.

We also collect out-of-fold predictions across all folds so the error analysis in Section 4 uses the whole dataset honestly.


In [4]:
# Metric helpers
def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())


def safe_roc_auc(y_true, scores):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, scores)


def safe_ap(y_true, scores):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return average_precision_score(y_true, scores)


def evaluate(name, y_true, scores):
    return {
        'model': name,
        'precision_at_10': precision_at_k(y_true, scores, k=10),
        'precision_at_50': precision_at_k(y_true, scores, k=50),
        'precision_at_100': precision_at_k(y_true, scores, k=100),
        'roc_auc': safe_roc_auc(y_true, scores),
        'average_precision': safe_ap(y_true, scores),
    }


# Models
models = {
    'logistic_regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    ]),
    'random_forest': RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_leaf=50,
        class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'gradient_boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        random_state=42
    )
}

fold_records = []
oof_scores = {name: np.full(len(y), np.nan) for name in list(models.keys()) + ['baseline_rule']}
importance_accum = {name: np.zeros(len(feature_cols)) for name in models.keys()}
coef_accum = {name: np.zeros(len(feature_cols)) for name in models.keys()}
n_folds = 0

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), 1):
    n_folds += 1
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Baseline rule on the held-out fold
    test_df = df_raw.iloc[test_idx].copy()
    baseline_score = np.where(
        test_df['gsc_impressions_90d'] >= 500,
        test_df['recent_share'] * np.log1p(test_df['gsc_impressions_90d']),
        0.0
    )
    oof_scores['baseline_rule'][test_idx] = baseline_score
    fold_records.append({**evaluate('baseline_rule', y_test, baseline_score), 'fold': fold})

    for name, model in models.items():
        if name == 'gradient_boosting':
            sample_weight = compute_sample_weight('balanced', y_train)
            model.fit(X_train, y_train, sample_weight=sample_weight)
        else:
            model.fit(X_train, y_train)

        proba = model.predict_proba(X_test)[:, 1]
        oof_scores[name][test_idx] = proba
        fold_records.append({**evaluate(name, y_test, proba), 'fold': fold})

        if name == 'logistic_regression':
            coef_accum[name] += model.named_steps['clf'].coef_[0]
        else:
            importance_accum[name] += model.feature_importances_

# Average importances / coefficients across folds
for name in models:
    if name == 'logistic_regression':
        coef_accum[name] /= n_folds
    else:
        importance_accum[name] /= n_folds

# Per-fold summary
fold_df = pd.DataFrame(fold_records)
cv_summary = fold_df.groupby('model').agg(
    precision_at_10=('precision_at_10', 'mean'),
    precision_at_50=('precision_at_50', 'mean'),
    precision_at_100=('precision_at_100', 'mean'),
    roc_auc=('roc_auc', 'mean'),
    average_precision=('average_precision', 'mean'),
).reset_index().sort_values('precision_at_50', ascending=False)

# Overall out-of-fold metrics (every prediction comes from a model that did not see that client)
oof_records = []
for name, scores in oof_scores.items():
    mask = ~np.isnan(scores)
    oof_records.append({**evaluate(name, y[mask], scores[mask]), 'model': name})
oof_df = pd.DataFrame(oof_records).sort_values('precision_at_50', ascending=False)

print('CV mean metrics per fold:')
print(cv_summary.round(4).to_string(index=False))
print('\nOverall out-of-fold metrics:')
print(oof_df.round(4).to_string(index=False))

# Save metrics
metrics_path = Path.cwd().parent / 'outputs' / 'w05_model_metrics.json'
metrics_path.parent.mkdir(parents=True, exist_ok=True)
with open(metrics_path, 'w') as f:
    json.dump({
        'cv_summary': cv_summary.to_dict(orient='records'),
        'oof_metrics': oof_df.to_dict(orient='records'),
        'fold_records': fold_df.to_dict(orient='records'),
        'feature_count': len(feature_cols),
        'features': feature_cols,
        'timestamp': datetime.now(timezone.utc).isoformat()
    }, f, indent=2)
print(f'Metrics saved to: {metrics_path}')


CV mean metrics per fold:
              model  precision_at_10  precision_at_50  precision_at_100  roc_auc  average_precision
  gradient_boosting             0.92            0.796             0.728   0.7305             0.4829
      random_forest             0.84            0.780             0.756   0.7244             0.4860
logistic_regression             0.54            0.576             0.616   0.6774             0.4304
      baseline_rule             0.50            0.428             0.464   0.5993             0.3512

Overall out-of-fold metrics:
              model  precision_at_10  precision_at_50  precision_at_100  roc_auc  average_precision
  gradient_boosting              1.0             0.92              0.93   0.7216             0.4584
      random_forest              0.8             0.82              0.77   0.7052             0.4707
      baseline_rule              0.3             0.48              0.45   0.5878             0.3423
logistic_regression              0.3        

## 4. Errors and interpretation

Use the best model from the CV comparison and its out-of-fold predictions to read the errors honestly.


In [5]:
# Best learned model by CV Precision@50
best_model_name = (
    cv_summary[cv_summary['model'] != 'baseline_rule']
    .sort_values('precision_at_50', ascending=False)
    .iloc[0]['model']
)
print(f'Best learned model by CV Precision@50: {best_model_name}')

# Build evaluation frame with original categoricals
eval_df = df_raw.copy()
eval_df['score'] = oof_scores[best_model_name]
eval_df['pred'] = (eval_df['score'] >= 0.5).astype(int)

# Feature importances or coefficients for the best model
if best_model_name == 'logistic_regression':
    fi_df = pd.DataFrame({
        'feature': feature_cols,
        'coef': coef_accum[best_model_name]
    })
    fi_df['abs_coef'] = fi_df['coef'].abs()
    fi_df = fi_df.sort_values('abs_coef', ascending=False)
    print('\nTop 10 Logistic Regression coefficients (by absolute value):')
    print(fi_df.head(10)[['feature', 'coef']].round(4).to_string(index=False))
else:
    fi_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': importance_accum[best_model_name]
    }).sort_values('importance', ascending=False)
    print('\nTop 10 feature importances:')
    print(fi_df.head(10).round(4).to_string(index=False))

# Concrete errors
false_pos = eval_df[(eval_df['growth_label'] == 0)].sort_values('score', ascending=False).head(3)
false_neg = eval_df[(eval_df['growth_label'] == 1)].sort_values('score', ascending=True).head(3)

print('\n3 false positives (predicted high, did not grow):')
print(false_pos[['content_hash_id', 'gsc_impressions_90d', 'gsc_impressions_last_30d',
                 'recent_share', 'gsc_avg_position_90d', 'content_age_days', 'score']].round(4).to_string(index=False))

print('\n3 false negatives (predicted low, actually grew):')
print(false_neg[['content_hash_id', 'gsc_impressions_90d', 'gsc_impressions_last_30d',
                 'recent_share', 'gsc_avg_position_90d', 'content_age_days', 'score']].round(4).to_string(index=False))

# Group-level error rates
def error_rates(group):
    fp = ((group['pred'] == 1) & (group['growth_label'] == 0)).mean()
    fn = ((group['pred'] == 0) & (group['growth_label'] == 1)).mean()
    return pd.Series({
        'n': len(group),
        'growth_rate': group['growth_label'].mean(),
        'false_positive_rate': fp,
        'false_negative_rate': fn
    })

print('\nError rates by main_intent:')
print(eval_df.groupby('main_intent').apply(error_rates, include_groups=False).round(3).to_string())

print('\nError rates by content_type:')
print(eval_df.groupby('content_type').apply(error_rates, include_groups=False).round(3).to_string())


Best learned model by CV Precision@50: gradient_boosting

Top 10 feature importances:
                     feature  importance
            content_age_days      0.2510
                recent_share      0.2376
                 gsc_ctr_90d      0.1037
                  word_count      0.1010
        gsc_avg_position_90d      0.0516
    gsc_impressions_last_30d      0.0420
log_gsc_impressions_last_30d      0.0414
   days_with_impressions_90d      0.0368
        sessions_organic_90d      0.0323
     log_gsc_impressions_90d      0.0181

3 false positives (predicted high, did not grow):
         content_hash_id  gsc_impressions_90d  gsc_impressions_last_30d  recent_share  gsc_avg_position_90d  content_age_days  score
content_ab17455adb87b6a1               1541.0                     156.0        0.1012               17.5240               447 0.9606
content_9a5547a73729eb51               1607.0                    1607.0        1.0000               32.1189                14 0.9597
content_ae897

## Self-check

Before you submit, confirm each line honestly:

- [x] Method choice explained: Logistic Regression, Random Forest, and Gradient Boosting for binary classification/ranking.
- [x] Split design is honest: 5-fold grouped CV by `client_hash_id`, stratified by label.
- [x] Baseline appears in the same comparison table as the models, same folds.
- [x] Engineered features are derived only from pre-decision data.
- [x] Notebook runs top to bottom with no errors (after first warehouse scan caches the feature vector).
- [x] No client names, URLs, or private queries anywhere.
- [x] Claims use careful words: observed, measured, directional, decision-support.
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
